# Discriminant Analysis for Alzheimer's Disease

This notebook mirrors the structure of INFO381A_LR.ipynb, but focuses on linear discriminant analysis (LDA), feature separation, and statistical interpretation.

Imports

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2, f_oneway
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')

DATA_PATH = Path('data/alzheimers_disease_data.csv')
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SYMPTOM_FEATURES = [
    'MMSE',
    'FunctionalAssessment',
    'MemoryComplaints',
    'BehavioralProblems',
    'ADL',
    'Confusion',
    'Disorientation',
    'PersonalityChanges',
    'DifficultyCompletingTasks',
    'Forgetfulness',
]

MEDICAL_AND_SYMPTOM_FEATURES = [
    'BMI',
    'Smoking',
    'AlcoholConsumption',
    'PhysicalActivity',
    'DietQuality',
    'SleepQuality',
    'FamilyHistoryAlzheimers',
    'CardiovascularDisease',
    'Diabetes',
    'Depression',
    'HeadInjury',
    'Hypertension',
    'SystolicBP',
    'DiastolicBP',
    'CholesterolTotal',
    'CholesterolLDL',
    'CholesterolHDL',
    'CholesterolTriglycerides',
    'MMSE',
    'FunctionalAssessment',
    'MemoryComplaints',
    'BehavioralProblems',
    'ADL',
    'Confusion',
    'Disorientation',
    'PersonalityChanges',
    'DifficultyCompletingTasks',
    'Forgetfulness',
]

Load dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

Basic overview

In [ ]:
df.info()
df.describe().T

Check for missing values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
missing.sort_values(ascending=False)

Class distribution

In [ ]:
print(df['Diagnosis'].value_counts())
print(df['Diagnosis'].value_counts(normalize=True))

plt.figure(figsize=(6, 4))
sns.countplot(x='Diagnosis', data=df, palette='magma')
plt.title('Diagnosis Distribution')
plt.xlabel('Diagnosis')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

Drop irrelevant columns and define feature groups

In [ ]:
df = df.drop(columns=['PatientID', 'DoctorInCharge'], errors='ignore')

target_col = 'Diagnosis'
binary_cols = [
    'Gender', 'Smoking', 'FamilyHistoryAlzheimers',
    'CardiovascularDisease', 'Diabetes', 'Depression',
    'HeadInjury', 'Hypertension',
    'MemoryComplaints', 'BehavioralProblems',
    'Confusion', 'Disorientation',
    'PersonalityChanges', 'DifficultyCompletingTasks',
    'Forgetfulness'
]
categorical_cols = ['Ethnicity', 'EducationLevel']
numeric_cols = [col for col in df.columns if col not in binary_cols + categorical_cols + [target_col]]

print('Numeric:', numeric_cols)
print('Binary:', binary_cols)
print('Categorical:', categorical_cols)

Correlation matrix

In [ ]:
plot_cols = [col for col in MEDICAL_AND_SYMPTOM_FEATURES if col in df.columns]
plt.figure(figsize=(13, 10))
sns.heatmap(df[plot_cols].corr(), cmap='magma', center=0)
plt.title('Correlation Matrix for LDA Features')
plt.tight_layout()
plt.show()

Prepare analysis helpers

In [ ]:
def make_pipeline():
    return Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('lda', LinearDiscriminantAnalysis()),
        ]
    )


def compute_wilks_lambda(X: pd.DataFrame, y: pd.Series):
    prep = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]
    )
    x_mat = prep.fit_transform(X)
    y_arr = np.asarray(y)
    classes = np.unique(y_arr)

    n_samples, n_features = x_mat.shape
    n_groups = len(classes)

    grand_mean = np.mean(x_mat, axis=0)
    centered = x_mat - grand_mean
    total_sscp = centered.T @ centered

    within_sscp = np.zeros_like(total_sscp)
    for cls in classes:
        x_group = x_mat[y_arr == cls]
        group_mean = np.mean(x_group, axis=0)
        group_centered = x_group - group_mean
        within_sscp += group_centered.T @ group_centered

    log_lambda = None
    ridge_used = None
    for ridge in (1e-10, 1e-8, 1e-6, 1e-4, 1e-2):
        identity = np.eye(n_features)
        sign_w, logdet_w = np.linalg.slogdet(within_sscp + ridge * identity)
        sign_t, logdet_t = np.linalg.slogdet(total_sscp + ridge * identity)
        if sign_w > 0 and sign_t > 0:
            log_lambda = float(logdet_w - logdet_t)
            ridge_used = ridge
            break

    if log_lambda is None:
        return {
            'wilks_lambda': None,
            'log_wilks_lambda': None,
            'chi_square_approx': None,
            'df': int(n_features * (n_groups - 1)),
            'p_value_approx': None,
            'ridge_used': None,
        }

    wilks_lambda = float(np.exp(log_lambda))
    df = int(n_features * (n_groups - 1))

    bartlett_factor = (n_samples - 1) - (n_features + n_groups) / 2
    chi_square = float(max(0.0, -bartlett_factor * log_lambda))
    p_value = float(1 - chi2.cdf(chi_square, df)) if df > 0 else None

    return {
        'wilks_lambda': wilks_lambda,
        'log_wilks_lambda': log_lambda,
        'chi_square_approx': chi_square,
        'df': df,
        'p_value_approx': p_value,
        'ridge_used': ridge_used,
    }


def run_lda_task(df: pd.DataFrame, target: str, features: list[str], task_name: str):
    available = [col for col in features if col in df.columns]
    data = df[available + [target]].dropna(subset=[target]).copy()

    X = data[available]
    y = data[target].astype(int)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pipeline = make_pipeline()
    y_pred = cross_val_predict(pipeline, X, y, cv=cv)

    accuracy = accuracy_score(y, y_pred)
    macro_f1 = f1_score(y, y_pred, average='macro')
    wilks_stats = compute_wilks_lambda(X, y)

    fitted = make_pipeline()
    fitted.fit(X, y)
    lda = fitted.named_steps['lda']

    explained_var = None
    cumsum_explained_var = None
    if hasattr(lda, 'explained_variance_ratio_'):
        explained_var = lda.explained_variance_ratio_.tolist()
        cumsum_explained_var = float(np.cumsum(lda.explained_variance_ratio_)[-1]) if len(lda.explained_variance_ratio_) > 0 else 0.0

    if y.nunique() == 2 and hasattr(lda, 'coef_'):
        feature_strength = np.abs(lda.coef_.ravel())
    else:
        feature_strength = np.mean(np.abs(lda.scalings_), axis=1)

    importance = (
        pd.DataFrame({'feature': available, 'importance_abs': feature_strength})
        .sort_values('importance_abs', ascending=False)
        .reset_index(drop=True)
    )

    importance_file = OUT_DIR / f'lda_feature_importance_{task_name}.csv'
    importance.to_csv(importance_file, index=False)

    return {
        'task': task_name,
        'target': target,
        'n_samples': int(len(data)),
        'n_features': int(len(available)),
        'random_state': 42,
        'cv_accuracy': float(accuracy),
        'cv_macro_f1': float(macro_f1),
        'explained_variance_ratio': explained_var,
        'total_explained_variance': cumsum_explained_var,
        'wilks_lambda': wilks_stats,
        'top_features': importance.head(10).to_dict(orient='records'),
        'importance_file': str(importance_file),
    }


def education_group_difference(df: pd.DataFrame, features: list[str]):
    target = 'EducationLevel'
    data = df[features + [target]].dropna(subset=[target]).copy()

    groups = sorted(data[target].astype(int).unique().tolist())
    results = []

    for feature in features:
        if feature not in data.columns:
            continue
        grouped = [data.loc[data[target] == g, feature].dropna().values for g in groups]
        if any(len(arr) < 2 for arr in grouped):
            continue
        if np.allclose(np.nanstd(data[feature].values), 0):
            continue

        f_stat, p_value = f_oneway(*grouped)

        grand_mean = data[feature].mean()
        ss_between = sum(len(arr) * (np.mean(arr) - grand_mean) ** 2 for arr in grouped)
        ss_total = np.sum((data[feature].values - grand_mean) ** 2)
        eta_sq = ss_between / ss_total if ss_total > 0 else 0.0

        group_means = {f'edu_{g}_mean': float(np.mean(arr)) for g, arr in zip(groups, grouped)}

        row = {
            'feature': feature,
            'f_stat': float(f_stat),
            'p_value': float(p_value),
            'eta_sq': float(eta_sq),
        }
        row.update(group_means)
        results.append(row)

    effects = pd.DataFrame(results).sort_values('eta_sq', ascending=False)
    effects_file = OUT_DIR / 'education_group_differences_anova.csv'
    effects.to_csv(effects_file, index=False)

    return {
        'n_features_tested': int(len(effects)),
        'top_effects': effects.head(10).to_dict(orient='records'),
        'effects_file': str(effects_file),
    }


def plot_top_importance(result: dict, title: str, top_n: int = 10):
    importance = pd.read_csv(result['importance_file']).head(top_n).iloc[::-1]
    plt.figure(figsize=(10, 6))
    plt.barh(importance['feature'], importance['importance_abs'], color='#3b6fb6')
    plt.title(title)
    plt.xlabel('Absolute discriminant strength')
    plt.tight_layout()
    plt.show()

Diagnosis from symptoms

In [ ]:
diagnosis_result = run_lda_task(
    df=df,
    target='Diagnosis',
    features=SYMPTOM_FEATURES,
    task_name='diagnosis_from_symptoms',
)

diagnosis_result

In [ ]:
plot_top_importance(diagnosis_result, 'Top LDA Features for Diagnosis from Symptoms')

Gender discrimination check

In [ ]:
gender_result = run_lda_task(
    df=df,
    target='Gender',
    features=MEDICAL_AND_SYMPTOM_FEATURES,
    task_name='gender_discrimination',
)

gender_result

Education level separation

In [ ]:
education_result = run_lda_task(
    df=df,
    target='EducationLevel',
    features=MEDICAL_AND_SYMPTOM_FEATURES,
    task_name='education_group_separation',
)

education_result

Education group differences (ANOVA)

In [ ]:
education_anova = education_group_difference(df, MEDICAL_AND_SYMPTOM_FEATURES)
education_anova

Summary table

In [ ]:
results_df = pd.DataFrame([
    {
        'task': diagnosis_result['task'],
        'target': diagnosis_result['target'],
        'n_samples': diagnosis_result['n_samples'],
        'n_features': diagnosis_result['n_features'],
        'cv_accuracy': diagnosis_result['cv_accuracy'],
        'cv_macro_f1': diagnosis_result['cv_macro_f1'],
        'wilks_lambda': diagnosis_result['wilks_lambda']['wilks_lambda'],
        'wilks_p_value': diagnosis_result['wilks_lambda']['p_value_approx'],
    },
    {
        'task': gender_result['task'],
        'target': gender_result['target'],
        'n_samples': gender_result['n_samples'],
        'n_features': gender_result['n_features'],
        'cv_accuracy': gender_result['cv_accuracy'],
        'cv_macro_f1': gender_result['cv_macro_f1'],
        'wilks_lambda': gender_result['wilks_lambda']['wilks_lambda'],
        'wilks_p_value': gender_result['wilks_lambda']['p_value_approx'],
    },
    {
        'task': education_result['task'],
        'target': education_result['target'],
        'n_samples': education_result['n_samples'],
        'n_features': education_result['n_features'],
        'cv_accuracy': education_result['cv_accuracy'],
        'cv_macro_f1': education_result['cv_macro_f1'],
        'wilks_lambda': education_result['wilks_lambda']['wilks_lambda'],
        'wilks_p_value': education_result['wilks_lambda']['p_value_approx'],
    },
]).round(4)

display(results_df)

Education effect sizes

In [ ]:
effects = pd.read_csv(education_anova['effects_file']).head(10).iloc[::-1]

plt.figure(figsize=(10, 6))
plt.barh(effects['feature'], effects['eta_sq'], color='#8a5cf6')
plt.title('Top Education-Level Separation Effects')
plt.xlabel('Eta squared')
plt.tight_layout()
plt.show()

Write report

In [ ]:
report = {
    'diagnosis_from_symptoms': diagnosis_result,
    'gender_discrimination': gender_result,
    'education_level_separation': education_result,
    'education_group_difference_tests': education_anova,
}

report_path = OUT_DIR / 'discriminant_analysis_report.json'
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(f'Wrote: {report_path}')

Conclusion

The LDA results summarize how strongly symptoms, clinical measurements, and demographic variables separate diagnosis, gender, and education groups in this dataset.